In [1]:

import sys
sys.path.append('/Users/brianlerner/Library/CloudStorage/OneDrive-DukeUniversity/Code/cerebra/src')

from openai import OpenAI
from tenacity import retry, wait_random_exponential, stop_after_attempt
from termcolor import colored
import json

from utils.viz import pretty_print_conversation
from viz.plot import plot_occurrence
from server.query import get_n_occurrences
from tools.tools import tools
import prompts.engine as PROMPTS




In [4]:

# to do
# 1. get embeddign funcs from notebook

GPT_MODEL = "gpt-3.5-turbo-0613"
# GPT_MODEL = "gpt-4"
EMBEDDING_MODEL = "text-embedding-ada-002"

# chat_response.choices[0] after tool call
# Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_JGeBCTAZqnC13FqECEpkCYuB', function=Function(arguments='{\n  "location": "Glasgow, Scotland",\n  "format": "celsius"\n}', name='get_current_weather'), type='function')]))

# chat_response.choices[0] after no tool call
# Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Sure, could you please provide me with the location?', role='assistant', function_call=None, tool_calls=None))



@retry(wait=wait_random_exponential(multiplier=1, max=40), stop=stop_after_attempt(3))
def chat_completion_request(conversation, tools=None, tool_choice=None, model=GPT_MODEL, reask=True, append=True):

    try:
        client = OpenAI()
        response = client.chat.completions.create(
                        model=model,
                        messages=conversation.conversation_history,
                        # stream=True,
                        tools=tools,
                        tool_choice=tool_choice,
                    )

        # make tool call if necessary
        full_message = response.choices[0].message
        if response.choices[0].finish_reason == "tool_calls":
            print("tool callinggg")
            function = full_message.tool_calls[0].function
            function_content = call_function(function, conversation)
            conversation.conversation_history.append(
                    {
                        "tool_call_id": full_message.tool_calls[0].id,
                        "role": "tool",
                        "name": function.name,
                        "content": function_content,
                    }
            )
            if reask:
                response = chat_completion_request(conversation)
                
        if append:
            # print("this one", response.choices[0].message)
            assistant_message = response.choices[0].message.content
            conversation.add_message("assistant", assistant_message)

        return response
        
    except Exception as e:
        print("Unable to generate ChatCompletion response")
        print(f"Exception: {e}")
        return e


def call_function(function, conversation):
    if function.name == "get_occurrence":
        conversation.add_message("system", PROMPTS.schema)
        # print("Getting occurrences")
        try:
            # print(function.arguments)
            function_arguments = json.loads(function.arguments)
            print('function args', function_arguments)
            n = get_n_occurrences(**function_arguments)
            print('got nnnnn')
            # fig, n = get_occurrence_information(**function_arguments)
            return n # how to show fig???
        except Exception as e:
            print(f"Function execution failed")
            print(f"Error message: {e}")
    else:
        raise Exception("Function does not exist and cannot be called")

def get_occurrence_information(variable):
    # fig = plot_occurrence(variable)
    n = get_n_occurrences(variable)
    return n


class Conversation:
    def __init__(self):
        self.conversation_history = []

    def add_message(self, role, content):
        message = {"role": role, "content": content}
        self.conversation_history.append(message)

    def display_conversation(self, detailed=False):
        role_to_color = {
            "system": "red",
            "user": "green",
            "assistant": "blue",
            "function": "magenta",
        }
        for message in self.conversation_history:
            print(
                colored(
                    f"{message['role']}: {message['content']}\n\n",
                    role_to_color[message["role"]],
                )
            )

### Conversation

questions = (i for i in PROMPTS.questions)
# Startthe conversation with a system message
conversation = Conversation()
conversation.add_message("system", PROMPTS.system)
# Get the initial response from the user (will integrate into streamlit after testing)
prompt = next(questions)
conversation.add_message("user", prompt)
chat_response = chat_completion_request(
    conversation, tools=tools
)
# assistant_message = chat_response.choices[0].message
pretty_print_conversation(conversation.conversation_history)
# display(Markdown(assistant_message))

tool callinggg
function args {'variable': 'Basketball Game'}
got nnnnn
Unable to generate ChatCompletion response
Exception: Error code: 400 - {'error': {'message': "'$.messages[3].content' is invalid. Please check the API reference: https://platform.openai.com/docs/api-reference.", 'type': 'invalid_request_error', 'param': None, 'code': None}}
Unable to generate ChatCompletion response
Exception: 'BadRequestError' object has no attribute 'choices'
system: You are Cerebra, a helpful assistant that pulls a user's personal data to help them know themselves better. You answer the user's question concisely and helpfully.
You always provide the stored name of the accessed data so the user can understand the process better.
Begin!

user: How many times does the Basketball Game occur?

system: The mapping between the dataset variable names and the categories they belong to:
{'elbow pain': 'extracted', 'Basketball Game': 'calendar_events', 'Computer Programming': 'calendar_events', 'Basketball

In [4]:
conversation.conversation_history

[{'role': 'system',
  'content': "You are Cerebra, a helpful assistant that pulls a user's personal data to help them know themselves better. You answer the user's question concisely and helpfully.\nYou always provide the stored name of the accessed data so the user can understand the process better.\nBegin!"},
 {'role': 'user', 'content': 'How many times does the Basketball Game occur?'}]

In [16]:
conversation = Conversation()
conversation.add_message("system", PROMPTS.system)
conversation.conversation_history

[{'role': 'system',
  'content': "You are Cerebra, a helpful assistant that pulls a user's personal data to help them know themselves better. You answer the user's question concisely and helpfully.\nYou always provide the stored name of the accessed data so the user can understand the process better.\nBegin!"}]